In [ ]:
# ==============================================================================
# PARALLEL ROLE-PLAYING DATES
# ==============================================================================
from helpers import setup_logger, write_gold_table, generate_date_dimension, generate_role_playing_date_dimensions, safe_count
from datetime import date

logger = setup_logger("parallel_role_playing_dates")

ROLE_PLAYING_DATES = [
    ("dim_service_date", "service_date_key"),
    ("dim_rental_date", "rental_date_key"),
    ("dim_return_date", "return_date_key"),
    ("dim_payment_date", "payment_date_key"),
    ("dim_payment_deadline_date", "payment_deadline_date_key"),
]

print("Row Counts (Before):")
print(f"dim_date: {safe_count(spark, 'dim_date')}")
print(f"dim_service_date: {safe_count(spark, 'dim_service_date')}")
print(f"dim_rental_date: {safe_count(spark, 'dim_rental_date')}")
print(f"dim_return_date: {safe_count(spark, 'dim_return_date')}")
print(f"dim_payment_date: {safe_count(spark, 'dim_payment_date')}")
print(f"dim_payment_deadline_date: {safe_count(spark, 'dim_payment_deadline_date')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))

if not spark.catalog.tableExists("wheelie.gold.dim_date"):
    logger.info("dim_date missing; generating base date dimension")
    dim_date = generate_date_dimension(
        spark,
        start_date=date(2000, 1, 1),
        end_date=date(2027, 12, 31),
    )
    write_gold_table(dim_date, "dim_date", mode="overwrite")
else:
    dim_date = spark.table("wheelie.gold.dim_date")

missing_role_dims = [
    (table_name, key_name)
    for table_name, key_name in ROLE_PLAYING_DATES
    if not spark.catalog.tableExists(f"wheelie.gold.{table_name}")
]

if missing_role_dims:
    logger.info(f"Creating {len(missing_role_dims)} role-playing date dimensions")
    role_playing_dims = generate_role_playing_date_dimensions(dim_date, missing_role_dims)
    for table_name, df in role_playing_dims:
        write_gold_table(df, table_name, mode="overwrite")
else:
    logger.info("All role-playing date dimensions already exist")

print("\nRow Counts (After):")
print(f"dim_date: {safe_count(spark, 'dim_date')}")
print(f"dim_service_date: {safe_count(spark, 'dim_service_date')}")
print(f"dim_rental_date: {safe_count(spark, 'dim_rental_date')}")
print(f"dim_return_date: {safe_count(spark, 'dim_return_date')}")
print(f"dim_payment_date: {safe_count(spark, 'dim_payment_date')}")
print(f"dim_payment_deadline_date: {safe_count(spark, 'dim_payment_deadline_date')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
